# TBot_SA1 策略学习 Notebook

本 notebook 用于理解 **TBot_SA1** 的完整数据流：从 LeRobot 数据集原始字段 → `DataTransformFn` 预处理链 → `forward` / `select_action` 推理。

> **环境要求**：`conda activate tbot_sa1`，并确保 `PYTHONPATH` 包含 `/vla/my_tbot/src`。
>
> **与 PI05 官方模式的差异**：TBot 不使用 `make_pre_post_processors`，而是在 `TBotSA1DatasetConfig.data_transforms` 中定义变换链，由 `TransformedLeRobotDataset` 在 `__getitem__` 时自动执行。

In [1]:
import os
import sys

# 离线模式：只用本地模型/数据集，禁止 HuggingFace 联网
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

# 若未 pip install -e /vla/my_tbot，需手动加入 src
if "/vla/my_tbot/src" not in sys.path:
    sys.path.insert(0, "/vla/my_tbot/src")

## 1. 模型加载

TBot 策略类为 `TBotSA1Policy`，checkpoint 目录需包含 `config.json` 和 `model.safetensors`。

**config 中说明了输入/输出字段**（state/action 会被 pad 到 `max_state_dim=32` / `max_action_dim=32`）。

In [2]:
import torch
from lerobot.configs.policies import PreTrainedConfig
from lerobot.policies.TBot_SA1.modeling_tbot_sa1 import TBotSA1Policy

# ---------- 路径配置（按需修改）----------
MODEL_ID = "/vla/.models/tbot_base"
QWEN3_VL_PATH = "/vla/.models/Qwen3-VL-2B-Instruct"
COSMOS_PATH = "/vla/.models/Cosmos-Tokenizer-CI8x8"
DA3_PATH = "/vla/.models/DA3-LARGE-1.1"
DA3_CODE_ROOT = "/vla/my_tbot/third_party/Depth-Anything-3"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 从 checkpoint 读取 config，并覆盖为本地路径
policy_cfg = PreTrainedConfig.from_pretrained(MODEL_ID, local_files_only=True)
policy_cfg.qwen3_vl_pretrained_path = QWEN3_VL_PATH
policy_cfg.cosmos_tokenizer_path_or_name = COSMOS_PATH
policy_cfg.da3_model_path_or_name = DA3_PATH
policy_cfg.da3_code_root = DA3_CODE_ROOT
policy_cfg.device = str(device)

policy = TBotSA1Policy.from_pretrained(
    MODEL_ID,
    config=policy_cfg,
    local_files_only=True,
).to(device).eval()

policy

/mnt/workspace/luyi/.cache/miniconda3/envs/tbot_sa1/lib/python3.10/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


[WARN ] Dependency `gsplat` is required for rendering 3DGS. Install via: pip install git+https://github.com/nerfstudio-project/gsplat.git@0b4dddf04cb687367602c01196913cde6a743d70
[INFO ] using MLP layer as FFN
Loading weights from local directory
Loading weights from local directory


TBotSA1Policy(
  (model): TBotSA1Model(
    (qwen3_vl_with_expert): Qwen3VLWithExpertModel(
      (und_expert): Qwen3VLForConditionalGeneration(
        (model): Qwen3VLModel(
          (visual): Qwen3VLVisionModel(
            (patch_embed): Qwen3VLVisionPatchEmbed(
              (proj): Conv3d(3, 1024, kernel_size=(2, 16, 16), stride=(2, 16, 16))
            )
            (pos_embed): Embedding(2304, 1024)
            (rotary_pos_emb): Qwen3VLVisionRotaryEmbedding()
            (blocks): ModuleList(
              (0-23): 24 x Qwen3VLVisionBlock(
                (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
                (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
                (attn): Qwen3VLVisionAttention(
                  (qkv): Linear(in_features=1024, out_features=3072, bias=True)
                  (proj): Linear(in_features=1024, out_features=1024, bias=True)
                )
                (mlp): Qwen3VLVisionMLP(
                  

In [3]:
print("=== input_features ===")
for k, v in policy.config.input_features.items():
    print(f"  {k}: {v}")

print("\n=== output_features ===")
for k, v in policy.config.output_features.items():
    print(f"  {k}: {v}")

print("\n=== 关键超参 ===")
print(f"chunk_size={policy.config.chunk_size}, n_action_steps={policy.config.n_action_steps}")
print(f"image_delta_indices={policy.config.image_delta_indices}")
print(f"dtype={policy.config.dtype}, num_inference_steps={policy.config.num_inference_steps}")

=== input_features ===
  observation.state: PolicyFeature(type=<FeatureType.STATE: 'STATE'>, shape=(32,))

=== output_features ===
  action: PolicyFeature(type=<FeatureType.ACTION: 'ACTION'>, shape=(32,))

=== 关键超参 ===
chunk_size=50, n_action_steps=50
image_delta_indices=[-15, 0, 15]
dtype=bfloat16, num_inference_steps=10


In [10]:
policy.config

TBotSA1Config(n_obs_steps=1, input_features={'observation.state': PolicyFeature(type=<FeatureType.STATE: 'STATE'>, shape=(32,))}, output_features={'action': PolicyFeature(type=<FeatureType.ACTION: 'ACTION'>, shape=(32,))}, device='cuda', use_amp=False, push_to_hub=False, repo_id='zaleni/TBot-SA1-Base', private=None, tags=None, license=None, pretrained_path=None, qwen3_vl_variant='qwen3_vl_28l', action_expert_variant='qwen3_28l', qwen3_vl_pretrained_path='/vla/.models/Qwen3-VL-2B-Instruct', dtype='bfloat16', chunk_size=50, n_action_steps=50, max_state_dim=32, max_action_dim=32, mask_action_dim_padding_loss=False, action_loss_valid_dim=None, num_inference_steps=10, time_sampling_beta_alpha=1.5, time_sampling_beta_beta=1.0, time_sampling_scale=0.999, time_sampling_offset=0.001, min_period=0.004, max_period=4.0, attention_mask_mode='default', image_resolution=(224, 224), image_delta_indices=[-15, 0, 15], empty_cameras=0, normalization_mapping={'VISUAL': <NormalizationMode.IDENTITY: 'IDENTI

TBotSA1Config(n_obs_steps=1, 
input_features={'observation.state': PolicyFeature(type=<FeatureType.STATE: 'STATE'>, shape=(32,))}, 
output_features={'action': PolicyFeature(type=<FeatureType.ACTION: 'ACTION'>, shape=(32,))}, device='cuda', use_amp=False, push_to_hub=False, repo_id='zaleni/TBot-SA1-Base', private=None, tags=None, license=None, pretrained_path=None, 

qwen3_vl_variant='qwen3_vl_28l', action_expert_variant='qwen3_28l',
qwen3_vl_pretrained_path='/vla/.models/Qwen3-VL-2B-Instruct', 

dtype='bfloat16', chunk_size=50, n_action_steps=50, max_state_dim=32, max_action_dim=32, mask_action_dim_padding_loss=False, action_loss_valid_dim=None, num_inference_steps=10, time_sampling_beta_alpha=1.5, time_sampling_beta_beta=1.0, time_sampling_scale=0.999, time_sampling_offset=0.001, min_period=0.004, max_period=4.0, attention_mask_mode='default', image_resolution=(224, 224), image_delta_indices=[-15, 0, 15], empty_cameras=0, normalization_mapping={'VISUAL': <NormalizationMode.IDENTITY: 'IDENTITY'>, 'STATE': <NormalizationMode.IDENTITY: 'IDENTITY'>, 'ACTION': <NormalizationMode.IDENTITY: 'IDENTITY'>}, gradient_checkpointing=False, compile_model=False, compile_mode='max-autotune', optimizer_lr=5e-05, optimizer_betas=(0.9, 0.95), optimizer_eps=1e-08, optimizer_weight_decay=0.01, optimizer_grad_clip_norm=1.0, scheduler_warmup_steps=2000, scheduler_decay_steps=300000, scheduler_decay_lr=1e-05, tokenizer_max_length=48, freeze_vision_encoder=False, train_expert_only=False, train_vlm_only=False, lora_modules=(), lora_unselected_mode='full', lora_targets=('attn', 'ffn'), lora_rank=16, lora_alpha=32.0, lora_rank_und=None, lora_alpha_und=None, lora_rank_gen=None, lora_alpha_gen=None, lora_rank_act=None, lora_alpha_act=None, lora_dropout=0.0, scale_factor=8, lambda_gen=0.01, cosmos_tokenizer_path_or_name='/vla/.models/Cosmos-Tokenizer-CI8x8', enable_3d_queries=True, num_3d_query_tokens=432, da3_alignment_mode='query_decoder', da3_query_resampler_layers=1, da3_query_resampler_ff_mult=1, query_layer_indices=(13, 19, 23, 27), da3_variant='auto', da3_teacher_layers=(11, 15, 19, 23), da3_query_dim=2048, da3_tokens_per_view=1296, da3_num_views=3, lambda_3d=0.01, da3_model_path_or_name='/vla/.models/DA3-LARGE-1.1', da3_model_name=None, da3_code_root='/vla/my_tbot/third_party/Depth-Anything-3', da3_teacher_process_res=504, da3_layer_weights=(1.0, 1.2, 1.4, 1.6), future_query_init_std=0.02, log_da3_teacher_timing=True)

## 2. 数据集处理

TBot 训练使用 `TransformedLeRobotDataset`：底层 `LeRobotDataset` 负责按 `delta_timestamps` 取时序帧，外层 transform 链负责归一化、图像重映射、Qwen3-VL tokenization 等。

In [4]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset

DATASET_PATH = "/vla/.data/adjust_bottle"  # <- 按需替换

# 原始数据集（未经 transform）
raw_ds = LeRobotDataset(
    repo_id=DATASET_PATH,
    video_backend="pyav",
)
raw_ds

LeRobotDataset({
    Repository ID: '/vla/.data/adjust_bottle',
    Number of selected episodes: '50',
    Number of selected samples: '10548',
    Features: '['observation.state', 'action', 'observation.images.cam_high', 'observation.images.cam_left_wrist', 'observation.images.cam_right_wrist', 'timestamp', 'frame_index', 'episode_index', 'index', 'task_index']',
})',

In [5]:
# 原始字段与 shape
for k in [
    "action",
    "observation.state",
    "observation.images.cam_high",
    "observation.images.cam_left_wrist",
    "observation.images.cam_right_wrist",
]:
    if k in raw_ds[0]:
        v = raw_ds[0][k]
        print(f"{k:45s} {tuple(v.shape) if hasattr(v, 'shape') else type(v)} {getattr(v, 'dtype', '')}")
    else:
        print(f"{k:45s} (not in dataset)")
print("task:", raw_ds[0].get("task"))

action                                        (14,) torch.float32
observation.state                             (14,) torch.float32
observation.images.cam_high                   (3, 480, 640) torch.float32
observation.images.cam_left_wrist             (3, 480, 640) torch.float32
observation.images.cam_right_wrist            (3, 480, 640) torch.float32
task: Use the left arm to pick up the narrow-necked bottle upright from the table.


### 2.1 构建与训练一致的数据集

`TBotSA1DatasetConfig.data_transforms.inputs` 定义了完整预处理链（见下一节逐步拆解）：

1. `DeltaActionTransformFn` — 可选，将 action 转为相对 state 的 delta
2. `ResizeImagesWithPadFn` — 图像 resize+pad 到 224×224
3. `RemapImageKeyTransformFn` — 相机键名 → `observation.images.image{0,1,2}`
4. `NormalizeTransformFn` — state/action 归一化（从 dataset.meta.stats 注入）
5. `ComposeFieldsTransform` — 合并多字段 state/action
6. `PadStateAndActionTransformFn` — pad 到 max_state_dim / max_action_dim
7. `Qwen3_VLProcessorTransformFn` — 图像 token + 语言 token
8. `UnifyTBotSA1InputsTransformFn` — 整理为 policy 期望的键集合

同时 `resolve_delta_timestamps` 会为 action 取 `chunk_size` 步、为每路相机取 `image_delta_indices=[-15,0,15]` 三帧。

In [6]:
from lerobot.configs.train import TrainPipelineConfig
from lerobot.datasets.factory import _build_single_dataset
from lerobot.policies.TBot_SA1.configuration_tbot_sa1 import TBotSA1DatasetConfig

dataset_cfg = TBotSA1DatasetConfig(
    repo_id=DATASET_PATH,
    qwen3_vl_processor_path=QWEN3_VL_PATH,
    video_backend="pyav",
)

train_cfg = TrainPipelineConfig(dataset=dataset_cfg, policy=policy_cfg, batch_size=1)

# 与 lerobot_train.py 中 make_dataset → _build_single_dataset 一致
transformed_ds, dataset_stats, robot_type = _build_single_dataset(
    train_cfg,
    DATASET_PATH,
    image_transforms=None,
    seed_offset=0,
)

print("robot_type:", robot_type)
print("transform pipeline:")
for i, step in enumerate(dataset_cfg.data_transforms.inputs):
    print(f"  [{i}] {step.__class__.__name__}")

transformed_ds

Hydrating transform NormalizeTransformFn with dataset.meta.stats (robot_type=aloha, resolved=aloha) and selected_keys (selected_keys=['observation.state', 'action'])
Hydrating transform ComposeFieldsTransform with mapping (robot_type=aloha, resolved=aloha)
Hydrating transform RemapImageKeyTransformFn with mapping (robot_type=aloha, resolved=aloha)
robot_type: aloha
transform pipeline:
  [0] ResizeImagesWithPadFn
  [1] RemapImageKeyTransformFn
  [2] NormalizeTransformFn
  [3] ComposeFieldsTransform
  [4] PadStateAndActionTransformFn
  [5] Qwen3_VLProcessorTransformFn
  [6] UnifyTBotSA1InputsTransformFn


TransformedLeRobotDataset({
    Repository ID: '/vla/.data/adjust_bottle',
    Number of selected episodes: '50',
    Number of selected samples: '10548',
    Features: '['observation.state', 'action', 'observation.images.cam_high', 'observation.images.cam_left_wrist', 'observation.images.cam_right_wrist', 'timestamp', 'frame_index', 'episode_index', 'index', 'task_index']',
})', (Transformed from LeRobotDataset)

注意 这里的图像数据是3帧

### 2.2 逐步观察 transform（类似 PI05 的 preprocess.steps）

下面手动逐步执行 transform，观察键名与张量 shape 的变化。

In [7]:
from dataclasses import replace
from lerobot.datasets.lerobot_dataset import LeRobotDatasetMetadata
from lerobot.datasets.factory import resolve_delta_timestamps
from lerobot.transforms.core import (
    compose,
    hydrate_normalize_transform,
    hydrate_compose_field_transform,
    hydrate_delta_action_transform,
    hydrate_remap_image_key_transform,
)

# 带 delta_timestamps 的原始帧（与 TransformedLeRobotDataset 底层一致）
ds_meta = LeRobotDatasetMetadata(DATASET_PATH)
delta_ts = resolve_delta_timestamps(policy_cfg, ds_meta)

base_ds = LeRobotDataset(
    repo_id=DATASET_PATH,
    video_backend="pyav",
    delta_timestamps=delta_ts,
)
frame = dict(base_ds[0])
print("原始 frame keys:", sorted(frame.keys()))
print("action shape:", tuple(frame["action"].shape))
print("observation.state shape:", tuple(frame["observation.state"].shape))

# hydrate 后的 transform 链
steps = list(dataset_cfg.data_transforms.inputs)
steps = hydrate_normalize_transform(steps, base_ds)
steps = hydrate_compose_field_transform(steps, base_ds)
steps = hydrate_delta_action_transform(steps, base_ds)
steps = hydrate_remap_image_key_transform(steps, base_ds)

data = frame
for i, step in enumerate(steps):
    data = step(data)
    tensor_shapes = {
        k: tuple(v.shape)
        for k, v in data.items()
        if hasattr(v, "shape")
    }
    print(f"\n--- step {i}: {step.__class__.__name__} ---")
    print("keys:", sorted(data.keys()))
    for k, shape in sorted(tensor_shapes.items()):
        print(f"  {k}: {shape}")

原始 frame keys: ['action', 'action_is_pad', 'episode_index', 'frame_index', 'index', 'observation.images.cam_high', 'observation.images.cam_high_is_pad', 'observation.images.cam_left_wrist', 'observation.images.cam_left_wrist_is_pad', 'observation.images.cam_right_wrist', 'observation.images.cam_right_wrist_is_pad', 'observation.state', 'robot_type', 'task', 'task_index', 'timestamp']
action shape: (50, 14)
observation.state shape: (14,)
Hydrating transform NormalizeTransformFn with dataset.meta.stats (robot_type=aloha, resolved=aloha) and selected_keys (selected_keys=['observation.state', 'action'])
Hydrating transform ComposeFieldsTransform with mapping (robot_type=aloha, resolved=aloha)
Hydrating transform RemapImageKeyTransformFn with mapping (robot_type=aloha, resolved=aloha)

--- step 0: ResizeImagesWithPadFn ---
keys: ['action', 'action_is_pad', 'episode_index', 'frame_index', 'index', 'observation.images.cam_high', 'observation.images.cam_high_is_pad', 'observation.images.cam_le

In [8]:
# 完整 transform 后的单样本（与 transformed_ds[0] 等价）
sample = transformed_ds[0]
print("预处理后 keys:", sorted(sample.keys()))
for k, v in sample.items():
    if hasattr(v, "shape"):
        print(f"  {k:40s} {tuple(v.shape)}")
    else:
        print(f"  {k:40s} {v}")

预处理后 keys: ['action', 'observation.attention_mask', 'observation.image_grid_thw', 'observation.images.image0', 'observation.images.image0_mask', 'observation.images.image1', 'observation.images.image1_mask', 'observation.images.image2', 'observation.images.image2_mask', 'observation.input_ids', 'observation.pixel_values', 'observation.state', 'sample.action_loss_mask']
  observation.state                        (32,)
  action                                   (50, 32)
  sample.action_loss_mask                  (1,)
  observation.images.image0                (3, 3, 224, 224)
  observation.images.image1                (3, 3, 224, 224)
  observation.images.image2                (3, 3, 224, 224)
  observation.images.image0_mask           ()
  observation.images.image1_mask           ()
  observation.images.image2_mask           ()
  observation.pixel_values                 (768, 1536)
  observation.image_grid_thw               (3, 3)
  observation.input_ids                    (246,)
  obse

### 2.3 组装 batch

训练时 `DataLoader` 默认 collate 会把样本 stack 成 batch。推理时 **不需要** `action` 输入，但 `forward` 训练需要 `(chunk_size, max_action_dim)` 的 action。

In [9]:
from torch.utils.data.dataloader import default_collate

batch = default_collate([sample])

def to_device(batch, device):
    return {
        k: v.to(device, non_blocking=True) if isinstance(v, torch.Tensor) else v
        for k, v in batch.items()
    }

batch = to_device(batch, device)
print("batch action shape:", batch["action"].shape)  # [B, chunk_size, max_action_dim]
print("batch observation.images.image0 shape:", batch["observation.images.image0"].shape)  # [B, T, C, H, W]

batch action shape: torch.Size([1, 50, 32])
batch observation.images.image0 shape: torch.Size([1, 3, 3, 224, 224])


## 3. 推理

`select_action(batch)` 是推理入口：内部调用 `predict_action_chunk` 做 flow-matching 去噪，再按 `n_action_steps` 逐步弹出 action。

> TBot 默认 `dtype=bfloat16`，notebook 推理建议包一层 `torch.autocast`，与训练 AMP 行为一致。

In [13]:
# 推理 batch 可以去掉 action（select_action 不读取 action）
infer_batch = {k: v for k, v in batch.items() if k != "action"}

with torch.inference_mode(), torch.autocast(device_type=device.type, dtype=torch.bfloat16):
    pred_action = policy.select_action(infer_batch)

# pred_action: [batch_size, action_dim]，已截断到真实维度（aloha=14，其余为 padding）
print("pred_action shape:", pred_action.shape)
print("pred_action:", pred_action)

pred_action shape: torch.Size([1, 32])
pred_action: tensor([[ 6.8017e-02, -9.4810e-02, -9.0081e-02,  2.0003e-02, -2.8989e-02,
          3.1162e-02,  9.7948e-01, -1.4131e-01, -1.3765e-01, -1.1822e-01,
          1.7223e-02,  9.4788e-03, -2.7193e-02, -8.5989e-02,  1.5897e-02,
          1.8167e-03, -2.3204e-03,  1.8607e-03, -4.3751e-03, -3.6355e-03,
          1.0170e-04, -2.9104e-03,  3.3925e-03, -8.0704e-04,  1.9507e-03,
         -1.0991e-03, -2.5141e-03, -4.6632e-03, -2.3489e-03,  1.5098e-03,
         -5.3329e-03,  3.5056e-03]], device='cuda:0')


In [14]:
# 一次预测完整 action chunk（chunk_size 步）
with torch.inference_mode(), torch.autocast(device_type=device.type, dtype=torch.bfloat16):
    action_chunk, recon_images = policy.predict_action_chunk(infer_batch)

original_action_dim = policy.config.output_features["action"].shape[0]
action_chunk = action_chunk[:, :, :original_action_dim]
print("action_chunk shape:", action_chunk.shape)  # [B, n_action_steps, action_dim]
print("first step action:", action_chunk[0, 0])

action_chunk shape: torch.Size([1, 50, 32])
first step action: tensor([ 7.2908e-02, -9.8582e-02, -9.0959e-02,  1.8846e-02, -2.9052e-02,
         2.6996e-02,  9.8065e-01, -1.4591e-01, -1.4518e-01, -1.2248e-01,
         1.9756e-02,  5.2259e-03, -3.3815e-02, -8.1620e-02,  8.4100e-03,
        -5.2297e-04, -4.3923e-03,  5.5635e-04,  6.9880e-04, -5.7648e-03,
        -1.2046e-03, -1.6688e-03,  3.1807e-03, -4.2045e-03, -1.9674e-03,
         2.6445e-03, -4.4298e-03, -5.6583e-03,  1.5762e-03,  5.1327e-04,
        -3.0479e-03,  5.1143e-03], device='cuda:0')


## 4. 训练 forward

`forward(batch)` 需要 batch 中包含 **完整 action chunk**（shape `[B, chunk_size, max_action_dim]`），由 `delta_timestamps` + transform 链保证。

返回 `(loss, loss_dict)`，其中总 loss = `loss_action + λ_gen·loss_gen + λ_3d·loss_3d`。

In [15]:
with torch.inference_mode(), torch.autocast(device_type=device.type, dtype=torch.bfloat16):
    loss, output_dict = policy.forward(batch)

print("loss:", loss.item())
print("output keys:", sorted(output_dict.keys()))
print("loss_action:", output_dict.get("loss_action"))
print("loss_gen:", output_dict.get("loss_gen"))
print("loss_3d:", output_dict.get("loss_3d"))

[INFO ] Selecting reference view using strategy: saddle_balanced
loss: 0.6927099823951721
output keys: ['loss', 'loss_3d', 'loss_3d_q13_t11', 'loss_3d_q19_t15', 'loss_3d_q23_t19', 'loss_3d_q27_t23', 'loss_action', 'loss_action_dim0', 'loss_action_dim1', 'loss_action_dim10', 'loss_action_dim11', 'loss_action_dim12', 'loss_action_dim13', 'loss_action_dim14', 'loss_action_dim15', 'loss_action_dim16', 'loss_action_dim17', 'loss_action_dim18', 'loss_action_dim19', 'loss_action_dim2', 'loss_action_dim20', 'loss_action_dim21', 'loss_action_dim22', 'loss_action_dim23', 'loss_action_dim24', 'loss_action_dim25', 'loss_action_dim26', 'loss_action_dim27', 'loss_action_dim28', 'loss_action_dim29', 'loss_action_dim3', 'loss_action_dim30', 'loss_action_dim31', 'loss_action_dim4', 'loss_action_dim5', 'loss_action_dim6', 'loss_action_dim7', 'loss_action_dim8', 'loss_action_dim9', 'loss_gen', 'time_3d_teacher_forward_s']
loss_action: 0.6701391935348511
loss_gen: 1.905845046043396
loss_3d: 0.351233541965

## 5. 数据流小结

| 阶段 | 主要键 | 典型 shape (aloha) |
|------|--------|-------------------|
| 原始数据集 | `observation.images.cam_*`, `observation.state`, `action` | 图像 `(3,H,W)`, state `(14,)`, action `(14,)` |
| + delta_timestamps | 同上 + 时序扩展 | action `(50,14)`, 图像 `(3,3,224,224)` 三帧 |
| + RemapImageKey | `observation.images.image{0,1,2}` | 统一 3 路相机命名 |
| + Pad | `observation.state`, `action` | pad 到 `(32,)` / `(50,32)` |
| + Qwen3VL Processor | `observation.pixel_values`, `input_ids`, `attention_mask` | VLM 输入 token |
| Policy 内部 | images, state, lang tokens | flow matching 采样 → action |

**与 PI05 对比**：
- PI05：`make_pre_post_processors` → `DataProcessorPipeline`（Rename / Normalize / Tokenizer / Device）
- TBot：`TransformedLeRobotDataset` + `DataTransformFn` 链（在 dataset 侧完成归一化，policy 侧无 postprocessor）
- TBot action 为 **chunk**（默认 50 步），PI05 可设 `chunk_size=1`

In [16]:
policy.config

TBotSA1Config(n_obs_steps=1, input_features={'observation.state': PolicyFeature(type=<FeatureType.STATE: 'STATE'>, shape=(32,))}, output_features={'action': PolicyFeature(type=<FeatureType.ACTION: 'ACTION'>, shape=(32,))}, device='cuda', use_amp=False, push_to_hub=False, repo_id='zaleni/TBot-SA1-Base', private=None, tags=None, license=None, pretrained_path=None, qwen3_vl_variant='qwen3_vl_28l', action_expert_variant='qwen3_28l', qwen3_vl_pretrained_path='/vla/.models/Qwen3-VL-2B-Instruct', dtype='bfloat16', chunk_size=50, n_action_steps=50, max_state_dim=32, max_action_dim=32, mask_action_dim_padding_loss=False, action_loss_valid_dim=None, num_inference_steps=10, time_sampling_beta_alpha=1.5, time_sampling_beta_beta=1.0, time_sampling_scale=0.999, time_sampling_offset=0.001, min_period=0.004, max_period=4.0, attention_mask_mode='default', image_resolution=(224, 224), image_delta_indices=[-15, 0, 15], empty_cameras=0, normalization_mapping={'VISUAL': <NormalizationMode.IDENTITY: 'IDENTI